<a href="https://colab.research.google.com/github/ghadaPoly/Projet_ML/blob/main/2_ML_cleaning_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⛳ Phase 2 : Nettoyage
Ce notebook présente l’étape de nettoyage des données (Data Cleaning) appliquée au dataset des patients diabétiques.
L’objectif est de préparer des données fiables, cohérentes et exploitables pour la modélisation en machine learning.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

### 1. Chargement des données
Nous commençons par charger le dataset en considérant que les valeurs "?" représentent des données manquantes.

 Cela permet de convertir automatiquement ces valeurs en NaN, facilitant leur traitement par la suite.

In [17]:
df = pd.read_csv("diabetic_data.csv",na_values="?")

print("Dimensions du dataset :", df.shape)

Dimensions du dataset : (101766, 50)


/tmp/ipykernel_5471/324205243.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("diabetic_data.csv",na_values="?")


### 2.Suppression des colonnes inutiles

Certaines variables ont été supprimées pour les raisons suivantes :

Identifiants uniques (encounter_id) → aucune valeur prédictive
Trop de valeurs manquantes (weight, medical_specialty)
Colonnes constantes (examide, citoglipton) → aucune information
Colonnes peu pertinentes (payer_code)

 Cette étape permet de réduire le bruit et d’améliorer la performance du modèle.

In [18]:
colonnes_a_supprimer = [
    'encounter_id',      # identifiant unique de la visite → pas prédictif
    'weight',            # 97% de valeurs manquantes → inutilisable
    'medical_specialty', # ~50% manquants
    'examide',           # colonne quasiment constante (une seule valeur)
    'citoglipton',       # même problème : colonne constante
    'payer_code',
]

df = df.drop(columns=colonnes_a_supprimer)

print(f"Colonnes après suppression : {df.shape[1]}")
print(f"Colonnes restantes : {df.columns.tolist()}")

Colonnes après suppression : 44
Colonnes restantes : ['patient_nbr', 'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [19]:
print(df.shape)

(101766, 44)


### 3.Gestion des doublons

Nous avons choisi de ne pas supprimer les visites multiples d’un même patient.

Justification :
Ces visites contiennent des informations temporelles importantes et enrichissent l’apprentissage du modèle.

In [20]:
# On NE supprime PAS les doublons par patient
print(f"Lignes conservées (avec visites multiples) : {len(df):,}")

print(f"Nombre de patients uniques : {df['patient_nbr'].nunique():,}")

Lignes conservées (avec visites multiples) : 101,766
Nombre de patients uniques : 71,518


### 4. Nettoyage de la variable gender

La valeur "Unknown/Invalid" a été supprimée car :

elle représente un très faible nombre de lignes
elle peut introduire du bruit dans le modèle

Résultat : une variable plus propre et cohérente.

In [21]:
print(f"\nValeurs de gender avant :\n{df['gender'].value_counts()}")
df = df[df['gender'] != 'Unknown/Invalid']

print(f"\nValeurs de gender après :\n{df['gender'].value_counts()}")



Valeurs de gender avant :
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

Valeurs de gender après :
gender
Female    54708
Male      47055
Name: count, dtype: int64


### 5.Transformation des variables diagnostiques

Les variables diag_1, diag_2, diag_3 contiennent des codes médicaux très détaillés (ICD-9).

**Problème :**

+ trop de valeurs uniques
+ difficile à exploiter directement

**Solution :**
Regrouper ces codes en grandes catégories médicales :

+ Circulatoire
+ Respiratoire
+ Digestif
+ Diabète
+ Blessure
etc.

**Avantage :**

+ réduction de la complexité
+ amélioration de la généralisation du modèle

In [22]:
def regrouper_diagnostic(code):

    if pd.isnull(code):
        return 'Autre'

    code = str(code)

    if code.startswith('V') or code.startswith('E'):
        return 'Autre'

    try:
        code_num = float(code)
    except ValueError:
        return 'Autre'

    if 390 <= code_num <= 459 or code_num == 785:
        return 'Circulatoire'
    elif 460 <= code_num <= 519 or code_num == 786:
        return 'Respiratoire'
    elif 520 <= code_num <= 579 or code_num == 787:
        return 'Digestif'
    elif code_num == 250:
        return 'Diabète'
    elif 800 <= code_num <= 999:
        return 'Blessure'
    elif 710 <= code_num <= 739:
        return 'Musculo-squelettique'
    elif 580 <= code_num <= 629 or code_num == 788:
        return 'Génito-urinaire'
    elif 140 <= code_num <= 239:
        return 'Néoplasme'
    else:
        return 'Autre'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(regrouper_diagnostic)

print("\nCatégories de diag_1 après regroupement :")
print(df['diag_1'].value_counts())


Catégories de diag_1 après regroupement :
diag_1
Circulatoire            30436
Autre                   26715
Respiratoire            14423
Digestif                 9475
Blessure                 6972
Génito-urinaire          5117
Musculo-squelettique     4957
Néoplasme                3433
Diabète                   235
Name: count, dtype: int64


### 6.Transformation des variables cibles

Les variables cibles ont été binarisées :

readmitted_bin :
+ 1 → patient réadmis en moins de 30 jours
+ 0 → sinon

change_bin :
+ 1 → traitement modifié
+ 0 → sinon

Les colonnes originales ont été supprimées pour éviter toute redondance.

In [23]:
#  On transforme nos deux cibles en 0/1 pour la classification.
#  readmitted : "<30" → 1 (ré-hospitalisé dans les 30 jours), sinon 0
#  change     : "Ch"  → 1 (traitement changé), sinon 0

df['readmitted_bin'] = (df['readmitted'] == '<30').astype(int)
df['change_bin']     = (df['change']     == 'Ch' ).astype(int)

# On supprime les colonnes originales (remplacées par les versions binaires)
df = df.drop(columns=['readmitted', 'change'])

print(f"\nreadmitted_bin : {df['readmitted_bin'].value_counts().to_dict()}")
print(f"change_bin     : {df['change_bin'].value_counts().to_dict()}")


readmitted_bin : {0: 90406, 1: 11357}
change_bin     : {0: 54754, 1: 47009}


In [24]:
print(df.shape)

(101763, 44)


### 7. Traitement des valeurs manquantes
Variables numériques :
*Remplacées par la médiane*

Pourquoi ?

+ robuste aux valeurs extrêmes
+ conserve la distribution globale

####Variables catégorielles

Trois stratégies ont été utilisées :

+ "Unknown" (ex: race)
→ pour conserver l’information de données manquantes
+ "Autre" (diagnostics)
→ logique dans un contexte médical
+ Mode (valeur la plus fréquente)
→ pour les autres variables

Cette approche évite d’introduire du biais inutile.

a discuter : Quand utiliser quoi ?
+  Mode : quand :
peu de NaN
variable stable
+ "Unknown" : quand :
NaN important
info inconnue importante

Pour certaines variables catégorielles comme race, nous avons préféré introduire une catégorie ‘Unknown’ afin de préserver l’information de données manquantes plutôt que de biaiser la distribution avec une imputation par le mode.”

In [25]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

cat_cols = [col for col in df.select_dtypes(include=['object']).columns
            if col not in ['readmitted_bin', 'change_bin']]

#  1. NUMÉRIQUES → médiane
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median = df[col].median()
        df[col] = df[col].fillna(median)
        print(f"{col} : NaN remplacés par médiane ({median})")

#  2. CATÉGORIELLES

# Colonnes où "Unknown" est plus logique que le mode
unknown_cols = ['race']  # tu peux ajouter d'autres si besoin

# Colonnes médicales où "Autre" est logique
autre_cols = ['diag_1', 'diag_2', 'diag_3']

for col in cat_cols:
    if df[col].isnull().sum() > 0:

        if col in unknown_cols:
            df[col] = df[col].fillna("Unknown")
            print(f"{col} : NaN remplacés par 'Unknown'")

        elif col in autre_cols:
            df[col] = df[col].fillna("Autre")
            print(f"{col} : NaN remplacés par 'Autre'")

        else:
            mode = df[col].mode().values[0]
            df[col] = df[col].fillna(mode)
            print(f"{col} : NaN remplacés par mode ({mode})")

race : NaN remplacés par 'Unknown'
max_glu_serum : NaN remplacés par mode (Norm)
A1Cresult : NaN remplacés par mode (>8)


In [26]:
print(df.shape)

(101763, 44)


### 8. Vérification finale

In [27]:
print("\n" + "=" * 55)
print("  RÉSUMÉ DU NETTOYAGE")
print("=" * 55)
print(f"  Lignes finales       : {len(df):,}")
print(f"  Colonnes finales     : {df.shape[1]}")
print(f"  Valeurs manquantes   : {df.isnull().sum().sum()}")
print(f"  Cible readmitted_bin : {df['readmitted_bin'].mean()*100:.1f}% positifs")
print(f"  Cible change_bin     : {df['change_bin'].mean()*100:.1f}% positifs")
print("\n  Prochaine étape : Encodage & Preprocessing")


  RÉSUMÉ DU NETTOYAGE
  Lignes finales       : 101,763
  Colonnes finales     : 44
  Valeurs manquantes   : 0
  Cible readmitted_bin : 11.2% positifs
  Cible change_bin     : 46.2% positifs

  Prochaine étape : Encodage & Preprocessing


In [28]:
df.shape

(101763, 44)

### 9. Sauvegarde du dataset

In [29]:
df.to_csv('diabetic_data_cleaned.csv', index=False)

from google.colab import files
files.download('diabetic_data_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
df.dtypes

,0
patient_nbr,int64
race,object
gender,object
age,object
admission_type_id,int64
discharge_disposition_id,int64
admission_source_id,int64
time_in_hospital,int64
num_lab_procedures,int64
num_procedures,int64
